In [2]:
import pandas as pd
from scipy.stats import kendalltau
import rbo

# Load your CSV files
df_1 = pd.read_csv("./data/1_all_dt.csv")
df_2 = pd.read_csv("./data/2_all_svm_rbf.csv")
df_3 = pd.read_csv("./data/3_anova_chi_gnb.csv")
df_4 = pd.read_csv("./data/4_base_agg_svm_poly_bagging.csv")
df_5 = pd.read_csv("./data/5_anova_chi_agg_boosting.csv")
df_6 = pd.read_csv("./data/6_anova_chi_agg_gnb.csv")

dfs = [df_1, df_2, df_3, df_4, df_5, df_6]

results = []

for i, df in enumerate(dfs, start=1):
    # ranked feature lists
    lime_features = df.reindex(df["LIME_Score"].abs().sort_values(ascending=False).index)["Feature"].tolist()
    shap_features = df.reindex(df["SHAP_Score"].abs().sort_values(ascending=False).index)["Feature"].tolist()
    fusion_features = df.sort_values(by="Final_Score", ascending=False)["Feature"].tolist()

    # common features for Kendall Tau
    common_fs = [f for f in fusion_features if f in shap_features]
    fusion_rank_fs = [fusion_features.index(f) for f in common_fs]
    shap_rank_fs = [shap_features.index(f) for f in common_fs]
    tau_fs, _ = kendalltau(fusion_rank_fs, shap_rank_fs)

    common_fl = [f for f in fusion_features if f in lime_features]
    fusion_rank_fl = [fusion_features.index(f) for f in common_fl]
    lime_rank_fl = [lime_features.index(f) for f in common_fl]
    tau_fl, _ = kendalltau(fusion_rank_fl, lime_rank_fl)

    common_sl = [f for f in shap_features if f in lime_features]
    shap_rank_sl = [shap_features.index(f) for f in common_sl]
    lime_rank_sl = [lime_features.index(f) for f in common_sl]
    tau_sl, _ = kendalltau(shap_rank_sl, lime_rank_sl)

    # RBO
    rbo_fs = rbo.RankingSimilarity(fusion_features, shap_features).rbo()
    rbo_fl = rbo.RankingSimilarity(fusion_features, lime_features).rbo()
    rbo_sl = rbo.RankingSimilarity(shap_features, lime_features).rbo()

    # Recall@K
    def recall_at_k(list1, list2, k):
        set1 = set(list1[:k])
        set2 = set(list2[:k])
        if len(set2) == 0:
            return 0
        return len(set1 & set2) / len(set2)

    row = {
        "Model": f"Model_{i}",
        "Kendall_Fusion_vs_SHAP": tau_fs,
        "Kendall_Fusion_vs_LIME": tau_fl,
        "Kendall_SHAP_vs_LIME": tau_sl,
        "RBO_Fusion_vs_SHAP": rbo_fs,
        "RBO_Fusion_vs_LIME": rbo_fl,
        "RBO_SHAP_vs_LIME": rbo_sl,
        "Recall@10_Fusion_vs_SHAP": recall_at_k(fusion_features, shap_features, 10),
        "Recall@10_Fusion_vs_LIME": recall_at_k(fusion_features, lime_features, 10),
        "Recall@10_SHAP_vs_LIME": recall_at_k(shap_features, lime_features, 10),
        "Recall@20_Fusion_vs_SHAP": recall_at_k(fusion_features, shap_features, 20),
        "Recall@20_Fusion_vs_LIME": recall_at_k(fusion_features, lime_features, 20),
        "Recall@20_SHAP_vs_LIME": recall_at_k(shap_features, lime_features, 20),
        "Recall@30_Fusion_vs_SHAP": recall_at_k(fusion_features, shap_features, 30),
        "Recall@30_Fusion_vs_LIME": recall_at_k(fusion_features, lime_features, 30),
        "Recall@30_SHAP_vs_LIME": recall_at_k(shap_features, lime_features, 30),
    }

    results.append(row)

results_df = pd.DataFrame(results)
print(results_df)

     Model  Kendall_Fusion_vs_SHAP  Kendall_Fusion_vs_LIME  \
0  Model_1                0.745455                0.745455   
1  Model_2                0.450549                0.824176   
2  Model_3               -0.294118                0.970588   
3  Model_4                0.636364                0.848485   
4  Model_5                0.928571                0.928571   
5  Model_6                0.642857                0.928571   

   Kendall_SHAP_vs_LIME  RBO_Fusion_vs_SHAP  RBO_Fusion_vs_LIME  \
0              0.490909            0.923124            0.881710   
1              0.274725            0.780524            0.854762   
2             -0.323529            0.438179            0.950980   
3              0.484848            0.782834            0.902447   
4              0.857143            0.968750            0.958333   
5              0.571429            0.832143            0.979167   

   RBO_SHAP_vs_LIME  Recall@10_Fusion_vs_SHAP  Recall@10_Fusion_vs_LIME  \
0          0.817821 

In [3]:
final_table = pd.DataFrame({
    "Fusion vs SHAP": [
        results_df["Kendall_Fusion_vs_SHAP"].mean(),
        results_df["RBO_Fusion_vs_SHAP"].mean(),
        results_df["Recall@10_Fusion_vs_SHAP"].mean(),
        results_df["Recall@20_Fusion_vs_SHAP"].mean(),
        results_df["Recall@30_Fusion_vs_SHAP"].mean(),
    ],
    "Fusion vs LIME": [
        results_df["Kendall_Fusion_vs_LIME"].mean(),
        results_df["RBO_Fusion_vs_LIME"].mean(),
        results_df["Recall@10_Fusion_vs_LIME"].mean(),
        results_df["Recall@20_Fusion_vs_LIME"].mean(),
        results_df["Recall@30_Fusion_vs_LIME"].mean(),
    ],
    "SHAP vs LIME": [
        results_df["Kendall_SHAP_vs_LIME"].mean(),
        results_df["RBO_SHAP_vs_LIME"].mean(),
        results_df["Recall@10_SHAP_vs_LIME"].mean(),
        results_df["Recall@20_SHAP_vs_LIME"].mean(),
        results_df["Recall@30_SHAP_vs_LIME"].mean(),
    ]
}, index=["Kendall Tau", "RBO", "Recall@10", "Recall@20", "Recall@30"])

print(final_table)

             Fusion vs SHAP  Fusion vs LIME  SHAP vs LIME
Kendall Tau        0.518280        0.874308      0.392587
RBO                0.787592        0.921233      0.710990
Recall@10          0.766667        1.000000      0.766667
Recall@20          1.000000        1.000000      1.000000
Recall@30          1.000000        1.000000      1.000000
